# AEROBOT – Fine-tune YOLOv8 deteksi orang (Google Colab)

1. Menu **Runtime → Change runtime type → GPU (T4)**.
2. Jalankan sel satu per satu.

In [ ]:
!nvidia-smi

## 1. Upload codebase
Upload `aerobot_vision.zip` lewat panel Files (kiri), lalu jalankan sel ini.

In [ ]:
!unzip -q -o aerobot_vision.zip
%cd aerobot_vision
!pip install -q -r requirements.txt

## 2. API key Roboflow
Ambil di Roboflow → Settings → API Keys. Lebih aman simpan di **Colab Secrets** (ikon kunci) dengan nama `ROBOFLOW_API_KEY`.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    from getpass import getpass
    os.environ['ROBOFLOW_API_KEY'] = getpass('ROBOFLOW_API_KEY: ')

## 3. Training
Ganti `WORKSPACE`, `PROJECT`, `VERSION` sesuai URL project Roboflow Anda:
`https://app.roboflow.com/WORKSPACE/PROJECT/VERSION`

In [ ]:
WORKSPACE = 'nama-workspace'
PROJECT = 'aerobot-person'
VERSION = 1

!python scripts/train_roboflow.py --workspace {WORKSPACE} --project {PROJECT} --version {VERSION} \
    --model yolov8n.pt --epochs 100 --imgsz 640 --batch 16 --export onnx

## 4. Lihat hasil
`results.png` = kurva loss/mAP, `confusion_matrix.png`, contoh prediksi `val_batch0_pred.jpg`.

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = f'runs/detect/aerobot_person/{f}'
    if os.path.exists(p): display(Image(p, width=900))

## 5. Unduh model
Salin ke folder `models/` di komputer robot, lalu ubah `model:` di `config.yaml`.

In [ ]:
from google.colab import files
files.download('runs/detect/aerobot_person/weights/best.pt')
files.download('runs/detect/aerobot_person/weights/best.onnx')